# SCNN v7 — FHT + ECA + Circular Aug + Option A/B

SCNN pipeline: FHT envelope, ECA attention, circular augmentation during training, label smoothing + noise.

In [1]:
import sys, math
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from config import RANDOM_SEED, N_CLASSES, MODELS_DIR, get_device
from src.experiment_runner import (
    get_splits, load_and_norm, split_cal_test,
    run_zero_shot, run_calibration, print_comparison,
    TEST_SUBJECTS, TRAIN_SUBJECTS,
)
from src.feature_extraction import fht_envelope_batch
from src.evaluation import measure_latency, print_latency

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = get_device()
splits = get_splits()
print(f'Device: {DEVICE}')

Device: mps


## Model

In [2]:
class ECA2d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        k = max(int(abs(math.log2(ch)/2+0.5)),3)
        k = k if k%2 else k+1
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1,1,k,padding=k//2,bias=False)
    def forward(self,x):
        b,c,h,w = x.size()
        w_ = self.gap(x).view(b,1,c)
        return x * torch.sigmoid(self.conv(w_)).view(b,c,1,1).expand_as(x)

class DSConv2d(nn.Module):
    def __init__(self,in_c,out_c,k=3,p=1):
        super().__init__()
        self.dw = nn.Conv2d(in_c,in_c,k,padding=p,groups=in_c)
        self.pw = nn.Conv2d(in_c,out_c,1)
    def forward(self,x): return self.pw(self.dw(x))

class SCNN_ECA(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            DSConv2d(1,48), nn.BatchNorm2d(48), nn.ReLU(), ECA2d(48), nn.MaxPool2d(2),
            DSConv2d(48,96), nn.BatchNorm2d(96), nn.ReLU(), ECA2d(96), nn.MaxPool2d(2),
            DSConv2d(96,192), nn.BatchNorm2d(192), nn.ReLU(), ECA2d(192), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(192,96), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(96, n_classes),
        )
    def forward(self,x): return self.classifier(self.features(x))
    def extract_feat(self,x):
        with torch.no_grad(): return nn.Flatten()(self.features(x))

print(f'Params: {sum(p.numel() for p in SCNN_ECA().parameters()):,}')

Params: 44,764


## Augmentation + training

In [3]:
def circular_augment(X, y, n_rot=8):
    N,C,T = X.shape
    Xa = np.empty((N*n_rot,C,T), dtype=X.dtype)
    ya = np.empty(N*n_rot, dtype=y.dtype)
    for s in range(n_rot):
        Xa[s*N:(s+1)*N] = np.roll(X, shift=s, axis=1)
        ya[s*N:(s+1)*N] = y
    idx = np.random.RandomState(RANDOM_SEED).permutation(len(ya))
    return Xa[idx], ya[idx]

class NoisyDS(Dataset):
    def __init__(self,X,y,std=0.1):
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long()
        self.std = std
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        x = self.X[i] + torch.randn_like(self.X[i])*self.std
        return x, self.y[i]

def apply_fht(X): return fht_envelope_batch(X)

def train_scnn(model, X_fht, y, n_epochs=50, lr=3e-3):
    model.to(DEVICE)
    loader = DataLoader(NoisyDS(X_fht, y), batch_size=256, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=len(loader))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    for ep in range(n_epochs):
        model.train()
        ls,c,t = 0,0,0
        for xb,yb in loader:
            xb,yb = xb.to(DEVICE),yb.to(DEVICE)
            out = model(xb); loss = crit(out,yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            ls += loss.item()*xb.size(0); c += (out.argmax(1)==yb).sum().item(); t += xb.size(0)
        if (ep+1)%10==0 or ep==0:
            print(f'  Epoch {ep+1:3d}/{n_epochs} — loss:{ls/t:.4f} acc:{c/t:.4f}')

@torch.no_grad()
def scnn_predict(X):
    model.eval()
    X_fht = apply_fht(X)
    Xt = torch.from_numpy(X_fht).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model(xb[0].to(DEVICE)).argmax(1).cpu().numpy() for xb in loader])

@torch.no_grad()
def scnn_features(X):
    model.eval()
    X_fht = apply_fht(X)
    Xt = torch.from_numpy(X_fht).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model.extract_feat(xb[0].to(DEVICE)).cpu().numpy() for xb in loader])

def scnn_finetune(X_cal, y_cal):
    F = scnn_features(X_cal)
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
    clf.fit(F, y_cal)
    def predict_ft(X): return clf.predict(scnn_features(X))
    return predict_ft

## Train on all data + circular augmentation

In [4]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Raw train: {X_train.shape}')

# FHT + circular augmentation
X_fht = apply_fht(X_train)
X_aug, y_aug = circular_augment(X_fht, y_train, n_rot=4)  # 4 rotations to keep manageable
print(f'After FHT + 4-rotation aug: {X_aug.shape[0]:,}')

model = SCNN_ECA().to(DEVICE)
train_scnn(model, X_aug, y_aug, n_epochs=50)

torch.save(model.state_dict(), MODELS_DIR / 'scnn_v7.pt')
print('Saved.')

Loading windows: 100%|██████████| 5790/5790 [00:07<00:00, 736.94it/s] 


Raw train: (651972, 8, 50)
After FHT + 4-rotation aug: 2,607,888
  Epoch   1/50 — loss:1.5338 acc:0.4561
  Epoch  10/50 — loss:1.3238 acc:0.5727
  Epoch  20/50 — loss:1.3133 acc:0.5784
  Epoch  30/50 — loss:1.2836 acc:0.5937
  Epoch  40/50 — loss:1.2434 acc:0.6150
  Epoch  50/50 — loss:1.2163 acc:0.6293
Saved.


## Option B — Zero-shot

In [5]:
print('Option B — Zero-shot:')
zero_results = run_zero_shot(scnn_predict, splits, norm_stats, name='SCNN+FHT')

Option B — Zero-shot:
  S1 zero-shot: 0.5112
  S2 zero-shot: 0.5033
  S3 zero-shot: 0.5040
  S4 zero-shot: 0.5934
  S5 zero-shot: 0.6843


## Option A — Calibration

Extract features from SCNN-ECA, train LogisticRegression on calibration data.

In [6]:
print('Option A — Calibration:')
cal_results = run_calibration(scnn_predict, scnn_finetune, splits, norm_stats, name='SCNN+FHT')

Option A — Calibration:
  S1 calibrated: 0.6830
  S2 calibrated: 0.7278
  S3 calibrated: 0.7561
  S4 calibrated: 0.7823
  S5 calibrated: 0.8261


## Latency

In [7]:
model.eval()
sample = X_train[:1]
sf = torch.from_numpy(apply_fht(sample)).float().unsqueeze(1).to(DEVICE)
for _ in range(10): _ = model(sf)
if DEVICE.type=='mps': torch.mps.synchronize()
def scnn_single(x):
    xf = apply_fht(x)
    xt = torch.from_numpy(xf).float().unsqueeze(1).to(DEVICE)
    with torch.no_grad(): out = model(xt)
    if DEVICE.type=='mps': torch.mps.synchronize()
    return out.argmax(1).cpu().numpy()
latency = measure_latency(scnn_single, sample, n_runs=500)
print_latency(latency, 'SCNN+FHT')


Latency — SCNN+FHT
  Mean:   2.85 ms
  Median: 2.79 ms
  P95:    4.15 ms
  <300ms: ✓


## Results

In [8]:
print_comparison(zero_results, cal_results, name='SCNN+FHT (v7)')


  SCNN+FHT (v7) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                51.12%       68.30%   +17.18%
S2                50.33%       72.78%   +22.45%
S3                50.40%       75.61%   +25.21%
S4                59.34%       78.23%   +18.89%
S5                68.43%       82.61%   +14.18%
